##### **TRAIN WITH sdv5 AND TEST WITH vqdm**

#### **1. INITIALIZATION & SPLIT DATA (6K sdv5 to train, 6k vqdm to test)**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import shutil
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# 1. Paths configuration
drive_csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
local_train_dir = "/content/Local_Train_sdv5"
local_test_dir = "/content/Local_Test_vqdm"
os.makedirs(local_train_dir, exist_ok=True)
os.makedirs(local_test_dir, exist_ok=True)

# 2. Define the missing copy_files function
def copy_files(df, target_dir, prefix):
    def copy_single(row_data):
        idx, row = row_data
        src = row['file_path']
        # Create a safe filename using prefix and index to avoid duplicates
        dest = os.path.join(target_dir, f"{prefix}_{idx}_{row['file_name']}")
        if not os.path.exists(dest):
            try:
                shutil.copy2(src, dest)
            except:
                return None
        return dest

    # Use 16 workers for maximum speed on Colab
    with ThreadPoolExecutor(max_workers=16) as executor:
        return list(tqdm(executor.map(copy_single, df.iterrows()), total=len(df)))

# 3. Balanced Sampling Logic (6K TOTAL: AI + NATURE)
df_full = pd.read_csv(drive_csv_path)

def get_balanced_sample(full_df, source_keyword, total_sample=6000):
    # Filter by source generator
    df_source = full_df[full_df['file_path'].str.contains(source_keyword, na=False)]

    # Separate AI and Nature based on path structure
    df_ai = df_source[df_source['file_path'].str.contains('/ai/', na=False)]
    df_nature = df_source[df_source['file_path'].str.contains('/nature/', na=False)]

    half = total_sample // 2

    # Balance the classes: 50% AI, 50% Nature
    s_ai = df_ai.sample(n=min(half, len(df_ai)), random_state=42)
    s_nature = df_nature.sample(n=min(half, len(df_nature)), random_state=42)

    # Combine and shuffle
    return pd.concat([s_ai, s_nature]).sample(frac=1, random_state=42)

# 4. Execute Sampling
print(f"🚀 Sampling 6K balanced images from SDv5 for Training...")
df_train_full = get_balanced_sample(df_full, 'imagenet_ai_0424_sdv5', 6000)

print(f"🚀 Sampling 6K balanced images from VQDM for Testing...")
df_test_full = get_balanced_sample(df_full, 'imagenet_ai_0419_vqdm', 6000)

# 5. Execute Copying (Now copy_files is defined!)
print(f"📦 Pulling Train set to local disk...")
df_train_full['file_path'] = copy_files(df_train_full, local_train_dir, "train")
df_train_full = df_train_full.dropna(subset=['file_path'])

print(f"📦 Pulling Test set to local disk...")
df_test_full['file_path'] = copy_files(df_test_full, local_test_dir, "test")
df_test_full = df_test_full.dropna(subset=['file_path'])

# 6. Save local CSVs
df_train_full.to_csv("/content/local_train.csv", index=False)
df_test_full.to_csv("/content/local_test.csv", index=False)

print(f"🎉 Done! Balanced Datasets Ready.")
print(f"Train (sdv5): {len(df_train_full)} images | Test (vqdm): {len(df_test_full)} images")

🚀 Sampling 6K balanced images from SDv5 for Training...
🚀 Sampling 6K balanced images from VQDM for Testing...
📦 Pulling Train set to local disk...


  0%|          | 0/6000 [00:00<?, ?it/s]

📦 Pulling Test set to local disk...


  0%|          | 0/6000 [00:00<?, ?it/s]

🎉 Done! Balanced Datasets Ready.
Train (sdv5): 6000 images | Test (vqdm): 6000 images


#### **2. LOAD MODEL ARCHITECTURE (Vision Encoder + Custom Linear Classification)**

In [4]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from transformers import CLIPVisionModel

model_path = "/content/drive/MyDrive/Model/clip_model"
# ========================================================
# DEFINING THE CUSTOM CLASSIFICATION ARCHITECTURE
# ========================================================
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super(ClassificationCLIP, self).__init__()

        print(f"Loading CLIP Vision Encoder from: {model_path}")
        # Load the base CLIP Vision model (The Backbone)
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)

        # Get the hidden size (usually 768 for ViT-B/32)
        hidden_size = self.vision_encoder.config.hidden_size

        print("Attaching Custom Linear Classification Head...")
        # Attach the "Tail": A Linear layer for binary classification (Real vs Fake)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        # 1. Forward pass through the CLIP Vision Encoder
        outputs = self.vision_encoder(pixel_values=pixel_values)

        # 2. Extract the pooled_output (The global feature vector of the image)
        pooled_output = outputs.pooler_output

        # 3. Pass through our custom linear head to get logits
        logits = self.classifier(pooled_output)

        return logits

# 1. Initialize your custom architecture from Cell 4
# Ensure 'device' and 'model_path' are defined as in your notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model = ClassificationCLIP(model_path).to(device)
my_model = my_model.to(device)
print(f"Model loaded and moved to {device}")


Loading CLIP Vision Encoder from: /content/drive/MyDrive/Model/clip_model


Loading weights:   0%|          | 0/391 [00:02<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.o

Attaching Custom Linear Classification Head...
Model loaded and moved to cuda


In [8]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# Standard CLIP preprocessing parameters
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)), # CLIP strictly requires 224x224 images
    transforms.ToTensor(),         # Convert PIL image to PyTorch Tensor
    transforms.Normalize(          # Normalization values specifically for CLIP
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

class FastDataset(Dataset):
    def __init__(self, csv_file, transform):
        """
        Args:
            csv_file (string): Path to the local CSV file.
            transform (callable): Image transformations (Resize, Normalize, etc.)
        """
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get image path and labels from CSV row
        row = self.df.iloc[idx]
        img_path = row['file_path']
        label = float(row['real_image'])
        group_name = row['label'] # Used for subgroup performance reporting

        try:
            # 2. Load and convert image to RGB
            with Image.open(img_path) as img:
                image = self.transform(img.convert('RGB'))
        except Exception as e:
            # 3. Fallback: Return a black image if the file is missing or corrupted
            image = torch.zeros((3, 224, 224))

        # Convert label to tensor (float32 is required for BCEWithLogitsLoss)
        label_tensor = torch.tensor([label], dtype=torch.float32)

        return image, label_tensor, group_name

# 2. Setup DataLoaders
train_loader = DataLoader(FastDataset("/content/local_train.csv", clip_transform),
                          batch_size=64, shuffle=True, num_workers=2, persistent_workers=True)
test_loader = DataLoader(FastDataset("/content/local_test.csv", clip_transform),
                         batch_size=64, shuffle=False, num_workers=2, persistent_workers=True)

# 3. Parameter Thawing (Unfreezing)
# Freeze everything first
for param in my_model.vision_encoder.parameters():
    param.requires_grad = False

# Unfreeze last 2 layers of CLIP Backbone
for layer in my_model.vision_encoder.vision_model.encoder.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze the custom Linear Head you attached
for param in my_model.classifier.parameters():
    param.requires_grad = True

print(f"🔥 Model initialized with {sum(p.numel() for p in my_model.parameters() if p.requires_grad):,} trainable parameters.")

🔥 Model initialized with 25,193,473 trainable parameters.


#### **3. TRAINING MODELS**

In [9]:
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from collections import defaultdict

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, my_model.parameters()), lr=2e-5)
scaler = torch.amp.GradScaler('cuda')

# Training on SDv5
my_model.train()
for epoch in range(3):
    bar = tqdm(train_loader, desc=f"Training SDv5 - Epoch {epoch+1}")
    for imgs, lbls, _ in bar:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            loss = criterion(my_model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        bar.set_postfix(loss=loss.item())

Training SDv5 - Epoch 1:   0%|          | 0/94 [00:00<?, ?it/s]

Training SDv5 - Epoch 2:   0%|          | 0/94 [00:00<?, ?it/s]

Training SDv5 - Epoch 3:   0%|          | 0/94 [00:00<?, ?it/s]

In [13]:
import torch
import os

# Save model to path
save_directory = "/content/drive/MyDrive/Model/clip_classification_weights"
save_filename = "model_train_only_sdv5.pth"
save_path = os.path.join(save_directory, save_filename)

# Create directory
os.makedirs(save_directory, exist_ok=True)

# Saving weights
print(f"Saving best model weights to: {save_path}")
torch.save(my_model.state_dict(), save_path)

print("Model saved successfully to Google Drive!")

Saving best model weights to: /content/drive/MyDrive/Model/clip_classification_weights/model_train_only_sdv5.pth
Model saved successfully to Google Drive!


#### **4. EVAL MODELS (ON BOTH SAME GENIMAGE DATASET AND UNSEEN GENIMAGE DATASET)**

In [10]:
# Cross-Evaluation on VQDM
my_model.eval()
results = defaultdict(lambda: [0, 0]) # [correct, total]
with torch.no_grad():
    for imgs, lbls, groups in tqdm(test_loader, desc="Testing on VQDM"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = (torch.sigmoid(my_model(imgs)) > 0.5).float()
        correct = (preds == lbls)
        for i in range(lbls.size(0)):
            results[groups[i]][1] += 1
            if correct[i].item(): results[groups[i]][0] += 1

print("\n" + "="*50)
print("FINAL CROSS-GENERATOR REPORT (Train: SDv5 | Test: VQDM)")
for g, data in results.items():
    acc = (data[0]/data[1])*100
    print(f"Group: {g:<20} | Accuracy: {acc:>6.2f}% ({data[0]}/{data[1]})")
print("="*50)

Testing on VQDM:   0%|          | 0/94 [00:00<?, ?it/s]


FINAL CROSS-GENERATOR REPORT (Train: SDv5 | Test: VQDM)
Group: ai                   | Accuracy:  28.47% (854/3000)
Group: nature               | Accuracy:  98.63% (2959/3000)


In [11]:
# ========================================================
# BLOCK 4: EVALUATION ON UNSEEN SDv5 SAMPLES (TRAIN WITH SDv5, EVAL WITH SDv5 --> Performance is very good)
# ========================================================

# 1. Identify indices of SDv5 images NOT used in Training
df_sdv5_all = df_full[df_full['file_path'].str.contains('imagenet_ai_0424_sdv5', na=False)]
train_indices = df_train_full.index
df_sdv5_unseen_pool = df_sdv5_all.drop(index=train_indices)

print(f"Total SDv5 images remaining (unseen): {len(df_sdv5_unseen_pool)}")

# 2. Sample balanced set (e.g., 2000 images) from the unseen pool
def sample_unseen_sdv5(df_pool, total_sample=2000):
    df_ai = df_pool[df_pool['file_path'].str.contains('/ai/', na=False)]
    df_nature = df_pool[df_pool['file_path'].str.contains('/nature/', na=False)]

    half = total_sample // 2
    s_ai = df_ai.sample(n=min(half, len(df_ai)), random_state=7)
    s_nature = df_nature.sample(n=min(half, len(df_nature)), random_state=7)

    return pd.concat([s_ai, s_nature]).sample(frac=1, random_state=7)

df_sdv5_test = sample_unseen_sdv5(df_sdv5_unseen_pool, 2000)
local_sdv5_test_dir = "/content/Local_Test_sdv5_unseen"
os.makedirs(local_sdv5_test_dir, exist_ok=True)

# 3. Pull files to local disk
print(f"🚀 Pulling {len(df_sdv5_test)} unseen SDv5 images for Evaluation...")
df_sdv5_test['file_path'] = copy_files(df_sdv5_test, local_sdv5_test_dir, "sdv5_unseen")
df_sdv5_test = df_sdv5_test.dropna(subset=['file_path'])
df_sdv5_test.to_csv("/content/local_sdv5_test_unseen.csv", index=False)

# 4. Create DataLoader
sdv5_test_loader = DataLoader(
    FastDataset("/content/local_sdv5_test_unseen.csv", clip_transform),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

# 5. Evaluation Loop
my_model.eval()
sdv5_results = defaultdict(lambda: [0, 0]) # [correct, total]

print("\n🔍 Running Evaluation on Unseen SDv5 Data...")
with torch.no_grad():
    for imgs, lbls, groups in tqdm(sdv5_test_loader, desc="Testing SDv5"):
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = (torch.sigmoid(my_model(imgs)) > 0.5).float()

        correct = (preds == lbls)
        for i in range(lbls.size(0)):
            sdv5_results[groups[i]][1] += 1
            if correct[i].item():
                sdv5_results[groups[i]][0] += 1

# 6. Final Report
print("\n" + "💎"*30)
print(f"🏆 IN-DISTRIBUTION PERFORMANCE (Unseen SDv5 Samples):")
for g, data in sdv5_results.items():
    accuracy = (data[0]/data[1])*100
    print(f"Group: {g:<15} | Accuracy: {accuracy:>6.2f}% ({data[0]}/{data[1]})")
print("💎"*30)

Total SDv5 images remaining (unseen): 10019
🚀 Pulling 2000 unseen SDv5 images for Evaluation...


  0%|          | 0/2000 [00:00<?, ?it/s]


🔍 Running Evaluation on Unseen SDv5 Data...


Testing SDv5:   0%|          | 0/32 [00:00<?, ?it/s]


💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎
🏆 IN-DISTRIBUTION PERFORMANCE (Unseen SDv5 Samples):
Group: nature          | Accuracy:  98.30% (983/1000)
Group: ai              | Accuracy:  97.70% (977/1000)
💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎
